# 🧠 Brain Tumor MRI Image Classification — ML/DL Capstone Project

| Field | Details |
|---|---|
| **Project Name** | Brain Tumor MRI Image Classification (Custom CNN + Transfer Learning + Streamlit Deployment) |
| **Project Type** | Classification (Deep Learning – Medical Image Classification) |
| **Contribution** | Individual *(edit if you worked in a team)* |
| **Team Member 1** | Suraj |
| **Team Member 2 / 3 / 4** | – |

## Project Summary

Brain tumors are among the most serious neurological conditions, and magnetic resonance imaging (MRI) is the primary non-invasive tool radiologists use to detect and characterise them. Manually reading large volumes of scans is time-consuming and subject to inter-observer variability, which motivates computer-aided triage tools that can flag the likely tumor type quickly and consistently. This project builds an end-to-end deep-learning pipeline that classifies brain MRI slices into multiple categories (typically glioma, meningioma, pituitary tumor and no tumor) and deploys it as a Streamlit web application for real-time inference.

The work follows the standard capstone structure. First, the image dataset is catalogued into a metadata table (path, class, original split, resolution, file size, brightness, contrast, entropy and an MD5 hash). Corrupt files and exact duplicate images are removed to prevent data leakage, and a stratified 70/15/15 train/validation/test split is created. An exploratory analysis following the UBM rule (Univariate, Bivariate, Multivariate) produces fifteen charts, including sample-image and mean-image visualisations, and three hypothesis tests (Kruskal–Wallis, Mann–Whitney U and chi-square) check whether image-level characteristics differ systematically between classes. This is a useful sanity check for dataset bias and shortcut learning.

Three models are implemented and compared: (1) a custom CNN built from scratch with four double-convolution blocks, batch normalisation, dropout and global average pooling; (2) MobileNetV2 transfer learning; and (3) EfficientNetB0 transfer learning. The pretrained networks first train a new classification head with a frozen backbone and are then fine-tuned by unfreezing the top layers at a low learning rate. Data augmentation (flip, rotation, zoom, contrast) and class weighting reduce overfitting and the effect of class imbalance. Each model is evaluated with accuracy, macro precision, recall, F1-score, ROC-AUC, confusion matrices and training curves. Random-search hyperparameter optimisation (learning rate, dropout, dense units) and stratified 3-fold cross-validation are used to tune and validate every model, and the improvement over the baseline is reported with updated metric score charts.

The final model is selected using validation macro-F1, so the test set stays untouched until final reporting. Grad-CAM heatmaps are generated for explainability, showing which image regions drive each prediction, which is essential for trust in a medical setting. The best model is saved in Keras format together with its class-name file, reloaded for a sanity check on unseen images, and served through a Streamlit app in which a user uploads an MRI image and receives the predicted tumor type with class probabilities and a low-confidence warning.

From a business perspective, recall on the tumor classes matters most because a missed tumor is far more costly than an unnecessary follow-up scan, while precision controls the workload caused by false alarms. Macro-F1 balances both across all classes and is therefore the main selection metric. The solution is not a diagnostic device; it is an educational decision-support prototype that shows how AI can speed up triage, reduce reader fatigue and standardise first-pass screening, while final decisions always remain with qualified clinicians. Future work includes external validation on scans from other hospitals, 3-D volumetric models, probability calibration and drift monitoring after deployment.

## GitHub Link
https://github.com/Suraj-05400/*your-repo-name* *(replace with the repository link of this project)*

## Problem Statement
Develop a deep-learning solution that automatically classifies brain MRI images into multiple tumor categories. The solution must (a) build a custom CNN from scratch, (b) improve performance with transfer learning on pretrained networks, (c) evaluate every model rigorously with business-relevant metrics, and (d) deploy the best model in a user-friendly Streamlit web application that gives real-time tumor-type predictions for uploaded MRI images.

## Real-Time Business Use Cases
* **AI-assisted medical diagnosis** – help radiologists classify brain tumors faster, cutting diagnostic turnaround time.
* **Early detection & patient triage** – automatically flag high-risk MRI scans for immediate specialist review.
* **Research & clinical trials** – segment patient datasets by tumor type for studies and trial recruitment.
* **Second-opinion AI systems** – support telemedicine and remote consultation in under-resourced regions.

## Dataset
Brain Tumor MRI Multi-Class Dataset (class = folder name, e.g. glioma / meningioma / pituitary / no tumor).

> **How to run:** put the dataset in `./data` (any layout where the **parent folder of each image is its class name**, e.g. `data/Training/glioma/img.jpg`) **or** set `BRAIN_DATA_DIR`. In Colab the notebook mounts Google Drive and uses `MyDrive/Tumour` (edit `DRIVE_DATA_PATH` in the data-loading cell). If no data folder is found it tries to download the public Kaggle dataset through `kagglehub`. Use a GPU runtime (Colab: *Runtime → Change runtime type → T4 GPU*) and choose *Runtime → Run all*.

# ***Let's Begin!***
## 1. Know Your Data
### Import Libraries & Global Configuration

In [ ]:
# ============================================================
# (Colab only) mount Google Drive and point the notebook at the dataset folder
# ============================================================
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DATA = "/content/drive/MyDrive/Tumour"      # <-- change to your dataset folder in Drive
    if os.path.isdir(DRIVE_DATA):
        os.environ["BRAIN_DATA_DIR"] = DRIVE_DATA
        print("Dataset folder set to:", DRIVE_DATA)
    else:
        print("Drive folder not found:", DRIVE_DATA, "- falling back to ./data or Kaggle download")
except ImportError:
    pass   # not running in Colab

In [ ]:
# ============================================================
# Import libraries & global configuration
# ============================================================
import os, sys, io, json, random, hashlib, warnings, subprocess, itertools, shutil, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
warnings.filterwarnings("ignore")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")   # keep TensorFlow logs quiet


def _ensure(pip_name, import_name):
    """Install a package on the fly if it is missing (keeps the notebook runnable in one go)."""
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])


for _pip, _imp in [("seaborn", "seaborn"), ("scikit-learn", "sklearn"), ("pillow", "PIL"),
                   ("scipy", "scipy"), ("joblib", "joblib")]:
    _ensure(_pip, _imp)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from PIL import Image
from scipy import stats
from IPython.display import display, Markdown
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, classification_report, confusion_matrix)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ---------------- Configuration (change here only) ----------------
SEED = 42
QUICK = os.environ.get("BRAIN_QUICK", "0") == "1"             # tiny smoke-test mode
PRETRAINED = os.environ.get("BRAIN_PRETRAINED", "1") == "1"   # use ImageNet weights
IMG_SIZE = int(os.environ.get("BRAIN_IMG_SIZE", 224))         # 224x224 as required by the project brief
BATCH = 32
EPOCHS = 1 if QUICK else 12          # baseline / tuned training epochs (early stopping applies)
FT_EPOCHS = 1 if QUICK else 6        # fine-tuning epochs for transfer-learning models
TUNE_EPOCHS = 1 if QUICK else 4      # epochs per hyper-parameter trial
N_TRIALS = 2 if QUICK else 3         # random-search trials per model
CV_FOLDS = 3
CV_EPOCHS = 1 if QUICK else 3        # epochs per cross-validation fold

random.seed(SEED); np.random.seed(SEED); keras.utils.set_random_seed(SEED)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100

print("TensorFlow:", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU") or "none (CPU – training will be slow)")
print(f"IMG_SIZE={IMG_SIZE} | EPOCHS={EPOCHS} | QUICK={QUICK} | PRETRAINED={PRETRAINED}")

### Dataset Loading
The loader (1) locates the dataset, (2) lists every image with its **class (= parent folder name)** and original split folder, and (3) extracts image-level metadata (resolution, file size, brightness, contrast, entropy, MD5 hash) in parallel. Unreadable files are kept with `NaN` values so that they show up in the missing-value analysis.

In [ ]:
# ============================================================
# Dataset loading
# ============================================================
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

# ---- Optional: Google Drive (Colab). Edit DRIVE_DATA_PATH to the folder that holds the class folders ----
DRIVE_DATA_PATH = "/content/drive/MyDrive/Tumour"
LOCAL_COPY = "/content/data_local"
try:
    from google.colab import drive            # only exists inside Colab
    drive.mount("/content/drive")
    if os.path.isdir(DRIVE_DATA_PATH) and not os.environ.get("BRAIN_DATA_DIR"):
        if not os.path.isdir(LOCAL_COPY):      # copy to local disk: reading thousands of files from Drive is slow
            print("Copying dataset from Drive to local disk (one-off, may take a few minutes)...")
            shutil.copytree(DRIVE_DATA_PATH, LOCAL_COPY)
        os.environ["BRAIN_DATA_DIR"] = LOCAL_COPY
except Exception as e:
    print("Google Drive not used:", type(e).__name__)


def count_images(folder):
    """Number of image files below a folder (0 if folder is invalid)."""
    return sum(1 for p in Path(folder).rglob("*") if p.suffix.lower() in IMG_EXT) if os.path.isdir(folder) else 0


def find_data_dir():
    """Locate the dataset: env variable -> common folders -> Kaggle download."""
    candidates = [os.environ.get("BRAIN_DATA_DIR"), "data", "Brain Tumor MRI Dataset",
                  "/content/data", "/kaggle/input/brain-tumor-mri-dataset"]
    for c in candidates:
        if c and count_images(c) > 0:
            return c
    try:  # last resort: download the public Kaggle dataset
        _ensure("kagglehub", "kagglehub")
        import kagglehub
        path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")
        if count_images(path) > 0:
            return path
    except Exception as e:
        print("Automatic download failed:", e)
    raise FileNotFoundError(
        "Dataset not found. Place images as data/<split>/<class>/<image> (or data/<class>/<image>), "
        "or set the BRAIN_DATA_DIR environment variable.")


DATA_DIR = find_data_dir()
print("Using dataset folder:", DATA_DIR)

SPLIT_MAP = {"training": "train", "testing": "test", "valid": "val", "validation": "val"}
rows = []
for p in Path(DATA_DIR).rglob("*"):
    if p.suffix.lower() in IMG_EXT:
        parts = p.relative_to(DATA_DIR).parts
        split = SPLIT_MAP.get(parts[0].lower(), parts[0].lower()) if len(parts) > 2 else "all"
        rows.append({"path": str(p), "label": p.parent.name, "orig_split": split})


def extract_metadata(path):
    """Read one image and return its metadata; failures return NaNs (corrupt/unreadable file)."""
    rec = dict(width=np.nan, height=np.nan, mode=None, file_kb=np.nan, mean_int=np.nan,
               std_int=np.nan, entropy=np.nan, dark_ratio=np.nan, md5=None, readable=False)
    try:
        raw = Path(path).read_bytes()
        rec["md5"] = hashlib.md5(raw).hexdigest()
        rec["file_kb"] = len(raw) / 1024
        with Image.open(io.BytesIO(raw)) as im:
            im.load()
            rec["width"], rec["height"], rec["mode"] = im.size[0], im.size[1], im.mode
            g = np.asarray(im.convert("L"), dtype=np.float32)
        hist = np.bincount(g.astype(np.uint8).ravel(), minlength=256) / g.size
        nz = hist[hist > 0]
        rec.update(mean_int=float(g.mean()), std_int=float(g.std()),
                   entropy=float(-(nz * np.log2(nz)).sum()), dark_ratio=float((g < 30).mean()), readable=True)
    except Exception:
        pass
    return rec


with ThreadPoolExecutor(max_workers=8) as ex:
    meta = pd.DataFrame(list(ex.map(extract_metadata, [r["path"] for r in rows])))
df_raw = pd.concat([pd.DataFrame(rows), meta], axis=1)
df_raw["aspect_ratio"] = df_raw["width"] / df_raw["height"]
print("Images found:", len(df_raw))

### Dataset First View

In [ ]:
# Dataset first look
df = df_raw.copy()
display(df.head())
display(df.tail(3))

### Dataset Rows & Columns count

In [ ]:
# Dataset rows & columns count
print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")

### Dataset Information

In [ ]:
# Dataset info
df.info()

### Duplicate Values
Exact duplicate images (same MD5 hash) are dangerous: if one copy lands in the training set and another in the test set the evaluation is inflated (**data leakage**).

In [ ]:
# Dataset duplicate value count (exact image duplicates via MD5 hash)
dup_mask = df["md5"].notna() & df.duplicated("md5", keep="first")
print("Exact duplicate images:", int(dup_mask.sum()), f"({dup_mask.mean():.2%})")
conflict = df[df["md5"].notna()].groupby("md5")["label"].nunique()
print("Hashes appearing under more than one class (label conflicts):", int((conflict > 1).sum()))

### Missing Values / Null Values

In [ ]:
# Missing values / null values count
missing = df.isnull().sum()
display(missing[missing > 0] if missing.sum() else pd.Series({"No missing values": 0}))

# Visualising the missing values
plt.figure(figsize=(10, 3.5))
sns.heatmap(df.isnull().sample(min(len(df), 1500), random_state=SEED).T, cbar=False, cmap="viridis")
plt.title("Missing-value map (yellow = missing; each column is an image)"); plt.xticks([])
plt.tight_layout(); plt.show()

### What did you know about your dataset?
The dataset is a collection of brain MRI slices stored as image files, where the folder name is the class label. Because images are not tabular, the notebook builds a **metadata table** (one row per image) containing the class, the original split folder, resolution, aspect ratio, file size, mean brightness, contrast (std), entropy, dark-pixel ratio and an MD5 hash. The code above reports the number of images, the number of exact duplicates, label conflicts and unreadable files; these counts decide the cleaning steps in *Data Wrangling*. Missing values can only occur when a file cannot be read (they are not imputable – a corrupt image is simply removed).
## 2. Understanding Your Variables

In [ ]:
# Dataset columns
print(list(df.columns))

In [ ]:
# Dataset describe
display(df.describe(include="all").T)

### Variables Description
| Variable | Meaning |
|---|---|
| `path` | Full path of the image file |
| `label` | **Target** – tumor class (parent folder name) |
| `orig_split` | Folder in which the image originally was (train/test/all) |
| `width`, `height`, `aspect_ratio` | Image resolution and shape |
| `mode` | Colour mode (L = grayscale, RGB, …) |
| `file_kb` | File size in kilobytes (proxy for detail / compression) |
| `mean_int` | Mean grayscale intensity (brightness) |
| `std_int` | Standard deviation of intensity (contrast) |
| `entropy` | Shannon entropy of the grey-level histogram (texture richness) |
| `dark_ratio` | Share of near-black pixels (amount of background) |
| `md5` | Hash used for duplicate detection |
| `readable` | `False` if the file could not be decoded |
### Check Unique Values for each variable

In [ ]:
# Check unique values for each variable
display(df.nunique().to_frame("unique_values"))
print("\nClasses:", sorted(df["label"].unique()))

## 3. Data Wrangling

In [ ]:
# ============================================================
# Data wrangling: make the dataset analysis-ready
# ============================================================
n0 = len(df)
df = df[df["readable"]].copy()                           # 1) drop unreadable / corrupt images
n1 = len(df)
df = df.drop_duplicates(subset="md5", keep="first")      # 2) drop exact duplicates (avoid leakage)
n2 = len(df)

# 3) engineered metadata helpers
px = df["width"] * df["height"]
df["res_bucket"] = pd.cut(px.rank(pct=True, method="first"), [0, 1/3, 2/3, 1],
                          labels=["Low-res", "Mid-res", "High-res"], include_lowest=True)
# 4) encode target
label_encoder = LabelEncoder()
df["label_id"] = label_encoder.fit_transform(df["label"])
CLASS_NAMES = list(label_encoder.classes_)
N_CLASSES = len(CLASS_NAMES)
df = df.reset_index(drop=True)

print(f"Original: {n0} | after removing unreadable: {n1} | after removing duplicates: {n2}")
print("Classes:", dict(zip(CLASS_NAMES, range(N_CLASSES))))
assert N_CLASSES >= 2, "At least two classes are required"
display(df.head())

### What all manipulations have you done and insights you found?
1. **Removed unreadable images** – a corrupt file can neither be viewed nor used for training; imputation makes no sense for images.
2. **Removed exact duplicates (MD5)** – duplicates inflate accuracy when copies fall in both train and test sets.
3. **Created `res_bucket`** (low/mid/high resolution tertiles) so resolution can be analysed as a categorical variable (Chart 11, Hypothesis 3).
4. **Label-encoded the target** (`label_id`) for `sparse_categorical_crossentropy`.

Insight: the numbers printed above show how many images were lost to cleaning; every remaining row is a unique, readable image with a known class, which is the minimum requirement for trustworthy evaluation.
## 4. Data Visualization, Storytelling & Experimenting with Charts
Charts follow the **UBM rule**: Univariate (1-4), Bivariate – numerical/categorical (5-7), numerical/numerical (8-9), categorical/categorical (10-11) and Multivariate (12-15). The `print` lines under each chart are *auto-generated insights* computed from your data.
### Chart - 1 : Class distribution (Univariate)

In [ ]:
# Chart 1 - class distribution (bar chart)
counts = df["label"].value_counts().reindex(CLASS_NAMES)
plt.figure(figsize=(7, 4))
ax = sns.countplot(data=df, x="label", order=CLASS_NAMES, hue="label", hue_order=CLASS_NAMES,
                   palette="viridis", legend=False)
for c in ax.containers:
    ax.bar_label(c)
plt.title("Number of images per class"); plt.xlabel("Tumor class"); plt.ylabel("Images")
plt.tight_layout(); plt.show()
print(f"Largest class: {counts.idxmax()} ({counts.max()}) | smallest: {counts.idxmin()} ({counts.min()}) | imbalance ratio = {counts.max()/counts.min():.2f}")

**1. Why did you pick the specific chart?** A bar chart is the clearest way to compare the *counts* of a few categorical groups and to spot class imbalance, which directly affects how the model must be trained.

**2. What is/are the insight(s) found from the chart?** The chart (and the printed imbalance ratio) shows how many images each tumor class has. A ratio close to 1 means a balanced problem; a large ratio means the model will see some classes far more often than others.

**3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth?** Yes – knowing the balance lets us choose class weights and the right metrics (macro-F1 / per-class recall) instead of plain accuracy. *Negative risk:* if a clinically important rare class is under-represented, an unweighted model would miss it, which in screening means missed tumors; this is why class weighting is used in training.
### Chart - 2 : Class share (Univariate)

In [ ]:
# Chart 2 - class share (donut chart)
plt.figure(figsize=(5.5, 5.5))
plt.pie(counts.values, labels=counts.index, autopct="%1.1f%%", startangle=90,
        wedgeprops=dict(width=0.45, edgecolor="white"))
plt.title("Share of each class in the dataset"); plt.tight_layout(); plt.show()
print("Class shares (%):", (counts / counts.sum() * 100).round(1).to_dict())

**1. Why did you pick the specific chart?** A donut chart expresses each class as a *part of the whole*, which is easy to communicate to non-technical stakeholders and complements the absolute counts of Chart 1.

**2. What is/are the insight(s) found from the chart?** The printed percentages show whether any class dominates the data (for example above ~35-40 %), or whether the data is close to uniform.

**3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth?** Positive: a roughly uniform share means the model will learn all tumor types fairly and reported accuracy is meaningful. Negative: a dominant class (such as "no tumor") can make the model look accurate by always predicting it, hiding poor recall on tumors – a patient-safety risk.
### Chart - 3 : Brightness distribution (Univariate)

In [ ]:
# Chart 3 - distribution of mean intensity (histogram + KDE)
plt.figure(figsize=(8, 4))
sns.histplot(df["mean_int"], bins=40, kde=True, color="#3b82f6")
plt.axvline(df["mean_int"].median(), color="red", ls="--", label=f"median = {df['mean_int'].median():.1f}")
plt.title("Distribution of mean image brightness"); plt.xlabel("Mean grayscale intensity (0-255)"); plt.legend()
plt.tight_layout(); plt.show()
print(f"Mean brightness: {df['mean_int'].mean():.1f} ± {df['mean_int'].std():.1f} | skewness = {df['mean_int'].skew():.2f}")

**1. Why did you pick the specific chart?** A histogram with KDE shows the full shape of a numeric variable (spread, skew, multiple peaks), which a single average cannot.

**2. What is/are the insight(s) found from the chart?** The spread of brightness reveals how different the acquisition settings are; several peaks or a long tail would indicate different scanners/protocols mixed in one dataset.

**3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth?** Positive: a wide spread justifies intensity normalisation and contrast augmentation so the model works on scans from other hospitals. Negative: if brightness differs strongly between sources, the network may learn the *source* instead of the *tumor*, and accuracy would fall on new hospitals.
### Chart - 4 : File-size distribution (Univariate)

In [ ]:
# Chart 4 - distribution of file size (histogram)
plt.figure(figsize=(8, 4))
sns.histplot(df["file_kb"], bins=40, kde=True, color="#10b981")
plt.title("Distribution of image file size"); plt.xlabel("File size (KB)")
plt.tight_layout(); plt.show()
print(f"File size: median {df['file_kb'].median():.1f} KB | 95th percentile {df['file_kb'].quantile(.95):.1f} KB | max {df['file_kb'].max():.1f} KB")

**1. Why did you pick the specific chart?** File size is a numeric variable; a histogram shows how storage/processing cost is distributed and whether a few huge files exist.

**2. What is/are the insight(s) found from the chart?** File size is a proxy for resolution and detail/compression. A right-skewed distribution means most files are small with some very large exceptions.

**3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth?** Positive: it shows that resizing all images to one size (done in the pipeline) is necessary and safe, keeping inference fast and cheap in deployment. Negative: very large files increase memory use and latency if they are not resized before inference.
### Chart - 5 : Brightness by class (Bivariate – Numerical vs Categorical)

In [ ]:
# Chart 5 - mean intensity by class (box plot)
plt.figure(figsize=(8, 4.5))
sns.boxplot(data=df, x="label", y="mean_int", order=CLASS_NAMES, hue="label", hue_order=CLASS_NAMES,
            palette="Set2", legend=False)
plt.title("Mean brightness by tumor class"); plt.xlabel("Class"); plt.ylabel("Mean intensity")
plt.tight_layout(); plt.show()
print("Median brightness per class:", df.groupby("label")["mean_int"].median().round(1).to_dict())

**1. Why did you pick the specific chart?** A box plot compares the median, spread and outliers of a numeric variable across categories side by side.

**2. What is/are the insight(s) found from the chart?** Overlapping boxes mean brightness alone cannot separate the classes; clearly separated medians mean brightness carries class information.

**3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth?** Positive: overlap implies the model must learn real tumor morphology, which generalises better. Negative: strongly separated brightness could be a *shortcut* (e.g. one class scanned with another protocol); the model would then fail on new data, so it must be checked with Grad-CAM later.
### Chart - 6 : Contrast by class (Bivariate – Numerical vs Categorical)

In [ ]:
# Chart 6 - contrast (std of intensity) by class (violin plot)
plt.figure(figsize=(8, 4.5))
sns.violinplot(data=df, x="label", y="std_int", order=CLASS_NAMES, hue="label", hue_order=CLASS_NAMES,
               palette="Set3", cut=0, inner="quartile", legend=False)
plt.title("Image contrast (std of intensity) by class"); plt.xlabel("Class"); plt.ylabel("Std of intensity")
plt.tight_layout(); plt.show()
print("Median contrast per class:", df.groupby("label")["std_int"].median().round(1).to_dict())

**1. Why did you pick the specific chart?** A violin plot adds the *density shape* to the quartiles, so differences in distribution (not only in median) between classes become visible.

**2. What is/are the insight(s) found from the chart?** The width of each violin shows how variable the contrast is inside a class; different shapes suggest different tissue patterns or acquisition styles per class.

**3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth?** Positive: it justifies contrast-based augmentation (`RandomContrast`) so the model is robust to scanner differences. Negative: if one class has systematically higher contrast because of the source, the model may over-rely on it, hurting real-world reliability.
### Chart - 7 : File size by class (Bivariate – Numerical vs Categorical)

In [ ]:
# Chart 7 - file size by class (box plot) - leakage / bias check
plt.figure(figsize=(8, 4.5))
sns.boxplot(data=df, x="label", y="file_kb", order=CLASS_NAMES, hue="label", hue_order=CLASS_NAMES,
            palette="pastel", legend=False)
plt.title("File size by class"); plt.xlabel("Class"); plt.ylabel("File size (KB)")
plt.tight_layout(); plt.show()
print("Median file size per class (KB):", df.groupby("label")["file_kb"].median().round(1).to_dict())

**1. Why did you pick the specific chart?** File size should be unrelated to the diagnosis. A box plot per class is a fast way to test whether it is (a metadata-leakage check).

**2. What is/are the insight(s) found from the chart?** Similar boxes mean file size does not reveal the class. Very different boxes would mean the classes came from different sources/compression settings.

**3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth?** Positive: confirming no strong leakage gives confidence that reported performance reflects medical content. Negative: if file size does separate classes, a model could "cheat" and performance in production would drop – a reputational and safety risk.
### Chart - 8 : Width vs Height (Bivariate – Numerical vs Numerical)

In [ ]:
# Chart 8 - width vs height coloured by class (scatter plot)
sample = df.sample(min(len(df), 3000), random_state=SEED)
plt.figure(figsize=(7, 5))
sns.scatterplot(data=sample, x="width", y="height", hue="label", hue_order=CLASS_NAMES, alpha=0.5, s=25)
plt.title("Image resolution (width vs height)"); plt.tight_layout(); plt.show()
print("Most common resolutions:", (df["width"].astype(int).astype(str) + "x" + df["height"].astype(int).astype(str)).value_counts().head(5).to_dict())

**1. Why did you pick the specific chart?** A scatter plot shows the relationship between two numeric variables and makes clusters of identical resolutions visible.

**2. What is/are the insight(s) found from the chart?** Dense clusters or vertical/horizontal lines reveal a few standard resolutions (and possibly several image sources); points far from the diagonal are non-square images.

**3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth?** Positive: it confirms that a resize step is needed and that a single square input size is appropriate, which simplifies deployment. Negative: if a class is concentrated in one resolution cluster, resizing may not remove that cue and the model could learn resolution instead of pathology.
### Chart - 9 : Brightness vs Contrast (Bivariate – Numerical vs Numerical)

In [ ]:
# Chart 9 - mean intensity vs contrast (scatter plot)
plt.figure(figsize=(7, 5))
sns.scatterplot(data=sample, x="mean_int", y="std_int", hue="label", hue_order=CLASS_NAMES, alpha=0.5, s=25)
plt.title("Brightness vs contrast"); plt.xlabel("Mean intensity"); plt.ylabel("Std of intensity")
plt.tight_layout(); plt.show()
print(f"Pearson correlation(mean_int, std_int) = {df['mean_int'].corr(df['std_int']):.2f}")

**1. Why did you pick the specific chart?** It reveals how two image-quality descriptors move together and whether classes form separate clouds.

**2. What is/are the insight(s) found from the chart?** The printed correlation quantifies the brightness-contrast relationship; heavily overlapping coloured clouds show that simple intensity statistics cannot classify the tumors and a deep model is needed.

**3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth?** Positive: overlap justifies investing in a CNN rather than a rule-based threshold system. Negative: clearly separable clouds would indicate easy shortcuts that may not hold on external data.
### Chart - 10 : Class vs original split folder (Bivariate – Categorical vs Categorical)

In [ ]:
# Chart 10 - class vs original split (heat map of counts)
ct = pd.crosstab(df["label"], df["orig_split"]).reindex(CLASS_NAMES)
plt.figure(figsize=(6, 4))
sns.heatmap(ct, annot=True, fmt="d", cmap="YlGnBu")
plt.title("Images per class in each original split folder"); plt.tight_layout(); plt.show()
print((ct.div(ct.sum(axis=0), axis=1) * 100).round(1).to_string())

**1. Why did you pick the specific chart?** A heat map of a contingency table is ideal for two categorical variables because every cell shows a count at once.

**2. What is/are the insight(s) found from the chart?** It shows whether the dataset's original train/test folders contain the same class mix. A different mix means the original split is not representative.

**3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth?** Positive: it justifies creating our own *stratified* split, so each subset has the same class proportions and evaluation is fair. Negative: using an unbalanced original test folder would give misleading metrics and could hide weak classes.
### Chart - 11 : Resolution bucket vs class (Bivariate – Categorical vs Categorical)

In [ ]:
# Chart 11 - class composition inside each resolution bucket (100 % stacked bar)
comp = pd.crosstab(df["res_bucket"], df["label"], normalize="index").reindex(columns=CLASS_NAMES) * 100
comp.plot(kind="bar", stacked=True, figsize=(7, 4.5), colormap="tab20c")
plt.title("Class composition per resolution bucket (%)"); plt.ylabel("%"); plt.xlabel("Resolution bucket")
plt.xticks(rotation=0); plt.legend(title="Class", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout(); plt.show()
print(comp.round(1).to_string())

**1. Why did you pick the specific chart?** A 100 % stacked bar compares proportions between groups even when group sizes differ.

**2. What is/are the insight(s) found from the chart?** If each bucket has about the same class mix, resolution is independent of the diagnosis; strongly unequal mixes show resolution is tied to the class.

**3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth?** Positive: independence means resizing all images to one size removes a possible bias. Negative: dependence would create a shortcut feature and a risk of poor performance on scans with other resolutions (tested statistically in Hypothesis 3).
### Chart - 12 : Sample MRI images per class (Multivariate – visual)

In [ ]:
# Chart 12 - sample images per class (image grid)
n_show = 4
fig, axes = plt.subplots(N_CLASSES, n_show, figsize=(2.6 * n_show, 2.6 * N_CLASSES))
axes = np.atleast_2d(axes)
for i, cls in enumerate(CLASS_NAMES):
    picks = df[df["label"] == cls].sample(min(n_show, (df["label"] == cls).sum()), random_state=SEED)
    for j in range(n_show):
        axes[i, j].axis("off")
        if j < len(picks):
            axes[i, j].imshow(Image.open(picks.iloc[j]["path"]).convert("L"), cmap="gray")
            if j == 0:
                axes[i, j].set_title(cls, loc="left", fontsize=11, fontweight="bold")
plt.suptitle("Random MRI samples per class", y=1.0); plt.tight_layout(); plt.show()

**1. Why did you pick the specific chart?** Medical images cannot be fully described by numbers; looking at real samples is essential to understand what the model has to learn and to detect label or quality problems.

**2. What is/are the insight(s) found from the chart?** The samples show differences in tumor position, shape and size, slice orientation (axial / sagittal / coronal) and brightness within the same class, i.e. the task has high intra-class variability.

**3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth?** Positive: the variability motivates augmentation and transfer learning, and visual confirmation of label quality reduces the risk of training on wrong labels. Negative: subtle differences between tumor types mean errors between similar classes are likely, so predictions must be presented as decision support and not as a diagnosis.
### Chart - 13 : Mean image per class (Multivariate – spatial)

In [ ]:
# Chart 13 - average image per class (spatial pattern)
fig, axes = plt.subplots(1, N_CLASSES, figsize=(3.2 * N_CLASSES, 3.4))
axes = np.atleast_1d(axes)
for ax, cls in zip(axes, CLASS_NAMES):
    paths = df[df["label"] == cls]["path"].sample(min(80, (df["label"] == cls).sum()), random_state=SEED)
    stack = np.stack([np.asarray(Image.open(p).convert("L").resize((128, 128)), dtype=np.float32) for p in paths])
    ax.imshow(stack.mean(axis=0), cmap="inferno"); ax.set_title(cls); ax.axis("off")
plt.suptitle("Mean MRI image per class (up to 80 images each)"); plt.tight_layout(); plt.show()

**1. Why did you pick the specific chart?** Averaging many images of a class shows the *typical spatial pattern* (where bright tissue, background and typical lesion locations are) – a multivariate view over thousands of pixels at once.

**2. What is/are the insight(s) found from the chart?** The mean images show where intensity is concentrated for every class; differences between them (for example the lesion locations that typically differ between tumor types, such as pituitary tumors near the base of the brain) indicate that spatial position is a learnable cue.

**3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth?** Positive: it supports using convolutional models, which exploit spatial structure, and warns us to keep augmentation moderate (small rotations/shifts) so that position cues are not destroyed. Negative: very similar mean images for two classes warn that those classes will be confused more often, which must be communicated to clinicians.
### Chart - 14 : Correlation Heatmap

In [ ]:
# Chart 14 - correlation heat map of numeric image descriptors
num_cols = ["width", "height", "aspect_ratio", "file_kb", "mean_int", "std_int", "entropy", "dark_ratio"]
corr = df[num_cols].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Correlation of image descriptors"); plt.tight_layout(); plt.show()
pairs = corr.where(np.triu(np.ones(corr.shape), 1).astype(bool)).stack().sort_values(key=np.abs, ascending=False)
print("Strongest correlations:\n", pairs.head(4).round(2).to_string())

**1. Why did you pick the specific chart?** A correlation heat map summarises all pairwise linear relationships between numeric variables in one view and exposes redundant (multicollinear) descriptors.

**2. What is/are the insight(s) found from the chart?** The strongest pairs printed above show which descriptors carry the same information (for instance brightness vs dark-pixel ratio, or resolution vs file size). These are metadata statistics, not CNN inputs – the CNN learns its own features from pixels.
### Chart - 15 : Pair Plot

In [ ]:
# Chart 15 - pair plot of key descriptors coloured by class
pp_cols = ["mean_int", "std_int", "entropy", "dark_ratio", "file_kb"]
pp = df.sample(min(len(df), 500), random_state=SEED)[pp_cols + ["label"]]
g = sns.pairplot(pp, hue="label", hue_order=CLASS_NAMES, diag_kind="hist", corner=True, plot_kws=dict(alpha=0.5, s=18))
g.fig.suptitle("Pair plot of image descriptors by class", y=1.02); plt.show()

**1. Why did you pick the specific chart?** A pair plot shows every pairwise relationship and every distribution at once, coloured by the target, which is the quickest multivariate check of class separability.

**2. What is/are the insight(s) found from the chart?** Strongly overlapping coloured clouds in all panels indicate that the classes are *not* separable by simple global statistics; the model has to learn local texture and shape, which is exactly what CNNs are designed for.
## 5. Hypothesis Testing
Based on the chart experiments, three hypothetical statements are tested at a significance level **α = 0.05**:

1. Mean brightness is different across the tumor classes (Chart 5).
2. Image contrast differs between "no tumor" images and tumor images (Chart 6).
3. Resolution bucket is associated with tumor class (Chart 11).

> With thousands of images even tiny differences become "significant", so effect sizes are printed next to each p-value.

In [ ]:
# Shared helpers for the hypothesis tests
ALPHA = 0.05
NEG_NAMES = {"notumor", "no_tumor", "notumour", "no", "normal", "healthy", "negative"}
NEG_CLASS = next((c for c in CLASS_NAMES if c.lower().replace(" ", "").replace("-", "") in NEG_NAMES), CLASS_NAMES[0])
print("Reference ('no tumor') class used in Hypothesis 2:", NEG_CLASS)

# Normality check (decides parametric vs non-parametric tests)
norm_sample = df["mean_int"].sample(min(len(df), 500), random_state=SEED)
print("Shapiro-Wilk on mean_int (n<=500): p =", f"{stats.shapiro(norm_sample)[1]:.4g}", "-> p<0.05 means NOT normal")

### Hypothetical Statement - 1
**1. Hypotheses**
- **H0:** The distribution of mean brightness is the same in all tumor classes.
- **H1:** At least one class has a different distribution of mean brightness.

**2. Perform an appropriate statistical test.**

In [ ]:
# Hypothesis 1 - Kruskal-Wallis H test
groups = [g["mean_int"].values for _, g in df.groupby("label")]
h_stat, p1 = stats.kruskal(*groups)
eps2 = (h_stat - len(groups) + 1) / (len(df) - len(groups))   # epsilon-squared effect size
print(f"H = {h_stat:.2f} | p-value = {p1:.4g} | effect size (epsilon^2) = {eps2:.3f}")
print("Decision:", "Reject H0 (brightness differs between classes)" if p1 < ALPHA else "Fail to reject H0")

**Which statistical test have you done to obtain P-Value?** Kruskal-Wallis H test (non-parametric one-way ANOVA on ranks).

**Why did you choose the specific statistical test?** There are more than two independent groups, and image statistics are not normally distributed (Shapiro-Wilk above) with unequal variances, so ANOVA assumptions do not hold. Kruskal-Wallis compares the groups' distributions without those assumptions.
### Hypothetical Statement - 2
**1. Hypotheses**
- **H0:** Image contrast (std of intensity) of "no tumor" images equals the contrast of tumor images.
- **H1:** The contrast of "no tumor" images differs from that of tumor images.

**2. Perform an appropriate statistical test.**

In [ ]:
# Hypothesis 2 - Mann-Whitney U test (reference class vs all tumor classes)
a = df.loc[df["label"] == NEG_CLASS, "std_int"].values
b = df.loc[df["label"] != NEG_CLASS, "std_int"].values
u_stat, p2 = stats.mannwhitneyu(a, b, alternative="two-sided")
auc_effect = u_stat / (len(a) * len(b))      # probability of superiority (0.5 = no effect)
print(f"U = {u_stat:.0f} | p-value = {p2:.4g} | probability of superiority = {auc_effect:.3f} (0.5 = no difference)")
print("Decision:", "Reject H0 (contrast differs)" if p2 < ALPHA else "Fail to reject H0")

**Which statistical test have you done to obtain P-Value?** Mann-Whitney U test (two-sided).

**Why did you choose the specific statistical test?** Two independent groups, a continuous non-normal variable: the Mann-Whitney U test is the non-parametric counterpart of the independent t-test and is robust to outliers and unequal group sizes.
### Hypothetical Statement - 3
**1. Hypotheses**
- **H0:** Resolution bucket and tumor class are independent.
- **H1:** Resolution bucket and tumor class are associated.

**2. Perform an appropriate statistical test.**

In [ ]:
# Hypothesis 3 - Chi-square test of independence (+ Cramer's V effect size)
table = pd.crosstab(df["res_bucket"], df["label"])
chi2, p3, dof, _ = stats.chi2_contingency(table)
cramers_v = np.sqrt(chi2 / (table.values.sum() * (min(table.shape) - 1)))
print(f"chi2 = {chi2:.2f} | dof = {dof} | p-value = {p3:.4g} | Cramer's V = {cramers_v:.3f}")
print("Decision:", "Reject H0 (resolution and class are associated)" if p3 < ALPHA else "Fail to reject H0")

**Which statistical test have you done to obtain P-Value?** Chi-square test of independence.

**Why did you choose the specific statistical test?** Both variables are categorical (resolution bucket, class) and the data is a contingency table of counts. Cramér's V shows whether a "significant" association is large enough to matter; a small V means resolution is not a practical shortcut.
## 6. Feature Engineering & Data Pre-processing
### 1. Handling Missing Values

In [ ]:
# Handling missing values & missing value imputation
print("Missing values after wrangling:", int(df[num_cols + ["label"]].isnull().sum().sum()))

**What all missing value imputation techniques have you used and why?** None was needed. The only possible "missing" data were unreadable images, which were removed in *Data Wrangling* because pixel data cannot be sensibly imputed and a wrongly imputed image could corrupt training.
### 2. Handling Outliers

In [ ]:
# Handling outliers: IQR rule on image descriptors (flag, inspect, keep)
rows_out = []
for c in ["mean_int", "std_int", "entropy", "dark_ratio", "file_kb"]:
    q1, q3 = df[c].quantile([.25, .75]); iqr = q3 - q1
    n_out = int(((df[c] < q1 - 1.5 * iqr) | (df[c] > q3 + 1.5 * iqr)).sum())
    rows_out.append({"feature": c, "outliers": n_out, "percent": round(100 * n_out / len(df), 2)})
display(pd.DataFrame(rows_out))

**What all outlier treatment techniques have you used and why?** Outliers were detected with the IQR rule but **kept**. In medical imaging an "unusual" image (very bright, very large, very dark) is often a legitimate scan variation or an unusual tumor, and removing it would make the model fragile on exactly those cases. Robustness is added instead through augmentation and normalisation.
### 3. Categorical Encoding

In [ ]:
# Encode categorical columns: target -> integer ids (sparse categorical cross-entropy)
mapping = dict(zip(CLASS_NAMES, label_encoder.transform(CLASS_NAMES)))
print("Label encoding:", mapping)

**What all categorical encoding techniques have you used & why?** *Label encoding* of the target. Together with `sparse_categorical_crossentropy` it is mathematically equivalent to one-hot encoding but uses less memory, and it works directly with Keras `class_weight`.
### 4. Textual Data Preprocessing

*Not applicable.* This is an image-classification project, so the NLP steps (contraction expansion, lower-casing, punctuation/URL/stop-word removal, rephrasing, tokenization, lemmatization, POS tagging and text vectorization) are not required. The image-equivalent steps – resizing, decoding, scaling and augmentation – are implemented in the next sections.

In [ ]:
# Textual data preprocessing is not applicable for image data
print("Textual data preprocessing: not applicable (image dataset).")

### 4. Feature Manipulation & Selection
#### 1. Feature Manipulation – data augmentation

In [ ]:
# Build the tf.data pipeline + stratified split first (needed for the visualisations below)
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["label_id"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label_id"], random_state=SEED)
train_df, val_df, test_df = [d.reset_index(drop=True) for d in (train_df, val_df, test_df)]

AUTOTUNE = tf.data.AUTOTUNE


def _load(path, label):
    """Read + decode + resize one image (stays in 0-255 range; scaling happens inside the model)."""
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    return tf.image.resize(img, (IMG_SIZE, IMG_SIZE)), label


def make_ds(paths, labels, training=False):
    """Create a batched tf.data pipeline. Only the training pipeline is shuffled."""
    ds = tf.data.Dataset.from_tensor_slices((np.asarray(paths), np.asarray(labels, dtype="int32")))
    if training:
        ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(_load, num_parallel_calls=AUTOTUNE)
    if training:                                  # brightness augmentation (+-25 grey levels), training data only
        ds = ds.map(lambda x, y: (tf.clip_by_value(tf.image.random_brightness(x, 25.0), 0.0, 255.0), y),
                    num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH).prefetch(AUTOTUNE)


def augmentation_block():
    """Augmentations are part of the model and active only during training."""
    return keras.Sequential([layers.RandomFlip("horizontal_and_vertical"),      # horizontal + vertical flips
                             layers.RandomRotation(0.08),                        # rotation (~ +-29 degrees)
                             layers.RandomZoom(0.1),                             # zoom
                             layers.RandomTranslation(0.08, 0.08),               # shifts
                             layers.RandomContrast(0.1)], name="augment")
# NOTE: brightness jitter is applied in the tf.data pipeline (make_ds) because the RandomBrightness layer
# cannot be serialised to the legacy .h5 format required by the project deliverables.


train_ds = make_ds(train_df["path"], train_df["label_id"], training=True)
val_ds = make_ds(val_df["path"], val_df["label_id"])
test_ds = make_ds(test_df["path"], test_df["label_id"])

# Visualise augmentation
img0 = _load(train_df.loc[0, "path"], 0)[0].numpy()
aug = augmentation_block()
fig, axes = plt.subplots(1, 5, figsize=(14, 3))
axes[0].imshow(img0.astype("uint8")); axes[0].set_title("original")
for k in range(1, 5):
    axes[k].imshow(np.clip(tf.image.random_brightness(aug(img0[None], training=True)[0], 25.0).numpy(), 0, 255).astype("uint8")); axes[k].set_title(f"augmented {k}")
for a_ in axes: a_.axis("off")
plt.tight_layout(); plt.show()

**Feature manipulation:** instead of hand-crafted features, the images are *resized* to 224x224 and *augmented* (horizontal + vertical flips, ±8 % rotation, 10 % zoom, 8 % shifts, brightness and contrast jitter) to create new training variants. Geometry changes are kept moderate so the tumor stays inside the image.
#### 2. Feature Selection

**What all feature selection methods have you used and why?** A CNN learns its own features from pixels, so classical feature selection does not apply. The equivalent is *transfer learning with partial fine-tuning*: the pretrained backbone is frozen at first (only the new head is learned), then **only the top 30 layers** are unfrozen. This limits trainable parameters (less over-fitting on a medium-sized medical dataset) and keeps the generic low-level features learned on ImageNet. Global average pooling and dropout further reduce over-fitting.

**Which all features you found important and why?** Important image regions are identified with **Grad-CAM** in Section 7 (explainability): it highlights the pixels that most strongly drive each prediction, and a good model should focus on the lesion, not on the background or the scan border.
### 5. Data Transformation

**Do you think that your data needs to be transformed?** Yes: images have different resolutions and may be 1-channel or 3-channel. They are (i) decoded to 3 channels, (ii) resized to `IMG_SIZE x IMG_SIZE` with bilinear interpolation and (iii) kept as `float32`. No log/power transformation is needed because the model works on pixels, not on skewed tabular variables.
### 6. Data Scaling

In [ ]:
# Data scaling: pixel range before and after model-specific scaling
xb, yb = next(iter(train_ds))
print("Raw batch range:", float(tf.reduce_min(xb)), "-", float(tf.reduce_max(xb)))
print("Custom CNN scaling  (x/255)        :", float(tf.reduce_min(xb / 255.)), "-", float(tf.reduce_max(xb / 255.)))
print("MobileNetV2 scaling (x/127.5 - 1)  :", float(tf.reduce_min(xb / 127.5 - 1)), "-", float(tf.reduce_max(xb / 127.5 - 1)))

**Which method have you used to scale your data and why?** Scaling is done *inside* each model so the saved model accepts raw 0-255 images (important for deployment – the Streamlit app needs no separate preprocessing code): `Rescaling(1/255)` for the custom CNN, `Rescaling(1/127.5, offset=-1)` (range [-1, 1]) for MobileNetV2 and the built-in normalisation of EfficientNetB0. Each pretrained network is fed the range it was trained with.
### 7. Dimensionality Reduction
**Do you think that dimensionality reduction is needed? Explain why.** Not for modelling – a CNN reduces dimensionality internally (convolutions, pooling, global average pooling). PCA is used here only to *visualise* how separable the classes are in a pretrained feature space.

In [ ]:
# Dimensionality reduction (PCA on pretrained MobileNetV2 embeddings - visualisation only)
emb_net = keras.applications.MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False,
                                         weights="imagenet" if PRETRAINED else None, pooling="avg")
emb_sample = train_df.sample(min(500, len(train_df)), random_state=SEED).reset_index(drop=True)
emb_ds = make_ds(emb_sample["path"], emb_sample["label_id"]).map(lambda x, y: (x / 127.5 - 1.0, y))
emb = emb_net.predict(emb_ds, verbose=0)
pca = PCA(n_components=min(30, emb.shape[0] - 1, emb.shape[1])).fit(emb)
proj = pca.transform(emb)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for cid, cname in enumerate(CLASS_NAMES):
    m = emb_sample["label_id"].values == cid
    ax[0].scatter(proj[m, 0], proj[m, 1], s=14, alpha=.6, label=cname)
ax[0].set_title("PCA of image embeddings (first 2 components)"); ax[0].legend()
ax[1].plot(np.cumsum(pca.explained_variance_ratio_), marker="o")
ax[1].set_title("Cumulative explained variance"); ax[1].set_xlabel("components")
plt.tight_layout(); plt.show()
del emb_net

**Which dimensionality reduction technique have you used and why?** PCA, because it is fast, linear and gives interpretable explained-variance numbers. It is used only for visualisation; the final models use the full feature maps.
### 8. Data Splitting

In [ ]:
# Data splitting: stratified 70 / 15 / 15
split_table = pd.DataFrame({"train": train_df["label"].value_counts(), "val": val_df["label"].value_counts(),
                            "test": test_df["label"].value_counts()}).reindex(CLASS_NAMES)
display(split_table); print("Totals:", split_table.sum().to_dict())

**What data splitting ratio have you used and why?** **70 % train / 15 % validation / 15 % test, stratified by class.** The validation set is used for early stopping, hyper-parameter search and model selection; the test set is touched only for final reporting. Stratification keeps the class proportions identical in all three subsets, and duplicates were removed beforehand so that no image appears in two subsets.
### 9. Handling Imbalanced Dataset

In [ ]:
# Handling imbalanced dataset: class weights
cw = compute_class_weight("balanced", classes=np.arange(N_CLASSES), y=train_df["label_id"].values)
CLASS_WEIGHT = {i: float(w) for i, w in enumerate(cw)}
print("Imbalance ratio (max/min class):", round(train_df["label"].value_counts().max() / train_df["label"].value_counts().min(), 2))
print("Class weights:", {CLASS_NAMES[i]: round(w, 3) for i, w in CLASS_WEIGHT.items()})

**Do you think the dataset is imbalanced? Explain why.** The imbalance ratio printed above (and Charts 1-2) answers this: a ratio close to 1 means the dataset is nearly balanced, a ratio well above ~1.5 means it is imbalanced.

**What technique did you use to handle the imbalance and why?** **Balanced class weights** in the loss function. They penalise mistakes on rare classes more heavily without throwing away majority-class images (undersampling) or creating synthetic, possibly unrealistic MRI images (oversampling/SMOTE). If the data is already balanced the weights are about 1.0 and have almost no effect.
## 7. ML Model Implementation
Common building blocks used by all three models: model factory, training loop, evaluation, plotting, hyper-parameter search and cross-validation.

In [ ]:
# ============================================================
# Reusable modelling utilities
# ============================================================
KINDS = ["custom_cnn", "mobilenetv2", "efficientnetb0"]
NAMES = {"custom_cnn": "Custom CNN", "mobilenetv2": "MobileNetV2 (transfer)", "efficientnetb0": "EfficientNetB0 (transfer)"}
DEFAULT_PARAMS = dict(lr=1e-3, dropout=0.3, units=128)
MODELS, RESULTS, HIST, BEST_PARAMS, CV_RESULTS, TRIALS = {}, {}, {}, {}, {}, {}


def make_base(kind):
    """Return (pretrained backbone, input-scaling layer) for a transfer-learning model."""
    w = "imagenet" if PRETRAINED else None
    shape = (IMG_SIZE, IMG_SIZE, 3)
    if kind == "mobilenetv2":
        return (keras.applications.MobileNetV2(input_shape=shape, include_top=False, weights=w),
                layers.Rescaling(1 / 127.5, offset=-1.0))
    if kind == "efficientnetb0":   # EfficientNet expects raw 0-255 pixels and normalises internally
        return (keras.applications.EfficientNetB0(input_shape=shape, include_top=False, weights=w),
                layers.Rescaling(1.0))
    raise ValueError(kind)


def compile_model(model, lr):
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="sparse_categorical_crossentropy", metrics=["accuracy"])


def build_model(kind, lr=1e-3, dropout=0.3, units=128):
    """Build + compile a model. Input = raw 0-255 image, output = class probabilities."""
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image")
    x = augmentation_block()(inp)
    if kind == "custom_cnn":
        x = layers.Rescaling(1 / 255.0)(x)
        for f in (32, 64, 128, 256):                      # 4 double-conv blocks
            for _ in range(2):
                x = layers.Conv2D(f, 3, padding="same", use_bias=False)(x)
                x = layers.BatchNormalization()(x)
                x = layers.ReLU()(x)
            x = layers.MaxPooling2D()(x)
            x = layers.Dropout(0.25)(x)
    else:
        base, scale = make_base(kind)
        base.trainable = False                            # stage 1: frozen backbone
        x = scale(x)
        x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(units, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(N_CLASSES, activation="softmax")(x)
    model = keras.Model(inp, out, name=kind)
    compile_model(model, lr)
    return model


def get_base(model):
    """The nested pretrained backbone (a keras.Model that is not the augmentation block)."""
    return [l for l in model.layers if isinstance(l, keras.Model) and l.name != "augment"][-1]


def unfreeze_top(model, n_layers=30, lr=1e-5):
    """Stage 2 of transfer learning: unfreeze the top layers (BatchNorm stays frozen) and recompile."""
    base = get_base(model)
    base.trainable = True
    for l in base.layers[:-n_layers]:
        l.trainable = False
    for l in base.layers:
        if isinstance(l, layers.BatchNormalization):
            l.trainable = False
    compile_model(model, lr)


def train(model, tr_ds, va_ds, epochs, tag=None):
    """Fit with EarlyStopping + ModelCheckpoint (best val_loss) + LR reduction; class weights handle imbalance."""
    cbs = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
           keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-7)]
    if tag:                                    # save the best epoch to disk (skipped for quick tuning/CV runs)
        os.makedirs("checkpoints", exist_ok=True)
        cbs.append(keras.callbacks.ModelCheckpoint(f"checkpoints/{tag}.keras", monitor="val_loss", save_best_only=True))
    return model.fit(tr_ds, validation_data=va_ds, epochs=epochs, class_weight=CLASS_WEIGHT, callbacks=cbs, verbose=2)


def merge_hist(*hs):
    out = {}
    for h in hs:
        for k, v in h.history.items():
            out.setdefault(k, []).extend(v)
    return out


def evaluate_model(model, ds, y_true):
    """Return metric dict, predicted labels and probabilities."""
    prob = model.predict(ds, verbose=0)
    pred = prob.argmax(axis=1)
    m = {"accuracy": accuracy_score(y_true, pred),
         "precision": precision_score(y_true, pred, average="macro", zero_division=0),
         "recall": recall_score(y_true, pred, average="macro", zero_division=0),
         "f1": f1_score(y_true, pred, average="macro", zero_division=0)}
    try:
        m["roc_auc"] = roc_auc_score(y_true, prob / prob.sum(1, keepdims=True), multi_class="ovr", average="macro")
    except Exception:
        m["roc_auc"] = np.nan
    return m, pred, prob


def store_results(name, model):
    mv, _, _ = evaluate_model(model, val_ds, val_df["label_id"].values)
    mt, pred, prob = evaluate_model(model, test_ds, test_df["label_id"].values)
    RESULTS[name] = {"val": mv, "test": mt, "test_pred": pred, "test_prob": prob}


def show_performance(name, title):
    """Evaluation metric score chart + confusion matrix + learning curves."""
    r, y = RESULTS[name], test_df["label_id"].values
    print(f"===== {title} (TEST set) =====")
    print(classification_report(y, r["test_pred"], target_names=CLASS_NAMES, zero_division=0))
    fig, ax = plt.subplots(1, 3, figsize=(18, 4.2))
    vals = {k: (0 if np.isnan(v) else v) for k, v in r["test"].items()}
    bars = ax[0].bar(list(vals.keys()), list(vals.values()), color=sns.color_palette("viridis", len(vals)))
    ax[0].bar_label(bars, fmt="%.3f"); ax[0].set_ylim(0, 1.1); ax[0].set_title("Evaluation metric score chart")
    cm = confusion_matrix(y, r["test_pred"], labels=range(N_CLASSES))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax[1], cbar=False)
    ax[1].set_title("Confusion matrix"); ax[1].set_xlabel("Predicted"); ax[1].set_ylabel("True")
    h = HIST.get(name)
    if h:
        ax[2].plot(h["accuracy"], label="train acc"); ax[2].plot(h["val_accuracy"], label="val acc")
        ax[2].plot(h["loss"], "--", label="train loss"); ax[2].plot(h["val_loss"], "--", label="val loss")
        ax[2].set_title("Learning curves"); ax[2].set_xlabel("epoch"); ax[2].legend()
    plt.suptitle(title, y=1.03); plt.tight_layout(); plt.show()


def step_baseline(kind):
    """Train the baseline (default hyper-parameters; frozen backbone for transfer models)."""
    model = build_model(kind, **DEFAULT_PARAMS)
    h = train(model, train_ds, val_ds, EPOCHS, tag=f"{kind}_base")
    name = f"{kind}_base"
    MODELS[name], HIST[name] = model, merge_hist(h)
    store_results(name, model)
    return model


def step_tune(kind):
    """Random-search HPO (lr, dropout, units) -> retrain best config (+ fine-tune for transfer models)."""
    space = list(itertools.product([1e-3, 5e-4, 1e-4], [0.2, 0.3, 0.5], [64, 128, 256]))
    random.Random(SEED).shuffle(space)
    trials = []
    for lr, do, un in space[:N_TRIALS]:
        m = build_model(kind, lr=lr, dropout=do, units=un)
        h = train(m, train_ds, val_ds, TUNE_EPOCHS)
        trials.append({"lr": lr, "dropout": do, "units": un, "val_accuracy": max(h.history["val_accuracy"])})
        del m
    tdf = pd.DataFrame(trials).sort_values("val_accuracy", ascending=False).reset_index(drop=True)
    TRIALS[kind] = tdf
    best = tdf.iloc[0]
    BEST_PARAMS[kind] = dict(lr=float(best["lr"]), dropout=float(best["dropout"]), units=int(best["units"]))
    print("Random-search trials:"); display(tdf)
    print("Best parameters:", BEST_PARAMS[kind])
    model = build_model(kind, **BEST_PARAMS[kind])
    h1 = train(model, train_ds, val_ds, EPOCHS, tag=f"{kind}_tuned")
    hs = [h1]
    if kind != "custom_cnn":                       # stage 2: fine-tune top layers
        unfreeze_top(model, 30, BEST_PARAMS[kind]["lr"] / 10)
        hs.append(train(model, train_ds, val_ds, FT_EPOCHS, tag=f"{kind}_tuned_finetune"))
    name = f"{kind}_tuned"
    MODELS[name], HIST[name] = model, merge_hist(*hs)
    store_results(name, model)
    return model


def step_cv(kind):
    """Stratified K-fold cross-validation of the tuned configuration on the training set."""
    skf = StratifiedKFold(CV_FOLDS, shuffle=True, random_state=SEED)
    X, y = train_df["path"].values, train_df["label_id"].values
    accs, f1s = [], []
    for i, (tr, te) in enumerate(skf.split(X, y), 1):
        m = build_model(kind, **BEST_PARAMS[kind])
        tr_ds, te_ds = make_ds(X[tr], y[tr], training=True), make_ds(X[te], y[te])
        m.fit(tr_ds, epochs=CV_EPOCHS, class_weight=CLASS_WEIGHT, verbose=0)
        pred = m.predict(te_ds, verbose=0).argmax(1)
        accs.append(accuracy_score(y[te], pred)); f1s.append(f1_score(y[te], pred, average="macro", zero_division=0))
        print(f"  fold {i}: accuracy={accs[-1]:.4f} | macro-F1={f1s[-1]:.4f}")
        del m
    CV_RESULTS[kind] = {"acc_mean": np.mean(accs), "acc_std": np.std(accs), "f1_mean": np.mean(f1s), "f1_std": np.std(f1s)}
    print(f"{CV_FOLDS}-fold CV -> accuracy {np.mean(accs):.4f} ± {np.std(accs):.4f} | macro-F1 {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")


def step_improvement(kind):
    """Baseline vs tuned metric score chart."""
    b, t = RESULTS[f"{kind}_base"]["test"], RESULTS[f"{kind}_tuned"]["test"]
    comp = pd.DataFrame({"Baseline": b, "Tuned": t}); comp["Improvement"] = comp["Tuned"] - comp["Baseline"]
    display(comp.round(4))
    comp[["Baseline", "Tuned"]].plot(kind="bar", figsize=(8, 3.8), color=["#94a3b8", "#16a34a"], rot=0)
    plt.ylim(0, 1.05); plt.title(f"{NAMES[kind]}: baseline vs tuned (test set)"); plt.tight_layout(); plt.show()

### ML Model - 1 : Custom CNN (built from scratch)

In [ ]:
# ML Model 1 - implementation: fit the algorithm + predict on the model (baseline)
step_baseline("custom_cnn")

#### 1. Explain the ML Model used and its performance using Evaluation metric Score Chart.
**Custom CNN:** four blocks of *(Conv-BN-ReLU) x 2 → MaxPool → Dropout* with 32-64-128-256 filters, followed by Global Average Pooling, a dense layer and a softmax output. Convolutions learn local patterns (edges → textures → shapes), batch normalisation stabilises training, dropout and augmentation reduce over-fitting and global average pooling keeps the parameter count low. It needs no pretrained weights but must learn everything from the MRI data.

In [ ]:
# Visualising evaluation metric score chart - Custom CNN baseline
show_performance("custom_cnn_base", "Custom CNN - baseline")

#### 2. Cross-Validation & Hyperparameter Tuning

In [ ]:
# ML Model 1 with hyper-parameter optimisation (random search) + cross-validation
step_tune("custom_cnn")
show_performance("custom_cnn_tuned", "Custom CNN - tuned")
step_cv("custom_cnn")

**Which hyperparameter optimization technique have you used and why?** **Random search** over learning rate, dropout and dense units (validation accuracy as score). With expensive deep models, random search explores the space much more efficiently than a full grid and needs only a few trials; the best configuration is retrained for the full number of epochs. **Stratified 3-fold cross-validation** then checks that the tuned configuration is stable across different data subsets (low standard deviation = reliable).

**Have you seen any improvement? Note down the improvement with updated Evaluation metric Score Chart.**

In [ ]:
# Improvement of the tuned Custom CNN over its baseline
step_improvement("custom_cnn")

The table and chart above give the exact change in accuracy, precision, recall, F1 and ROC-AUC. A small or even negative change means the default configuration was already close to optimal for this architecture; the cross-validation standard deviation tells how much of any change is noise.
### ML Model - 2 : MobileNetV2 (transfer learning)

In [ ]:
# ML Model 2 - implementation: frozen MobileNetV2 backbone + new head (baseline)
step_baseline("mobilenetv2")

#### 1. Explain the ML Model used and its performance using Evaluation metric Score Chart.
**MobileNetV2** is a lightweight CNN pretrained on ImageNet that uses depth-wise separable convolutions and inverted residual blocks, giving high accuracy with few parameters and fast inference (ideal for a web app). The baseline keeps the backbone frozen and trains only the new classification head (global average pooling → dropout → dense → softmax); it re-uses generic visual features (edges, textures, shapes) that transfer well to MRI.

In [ ]:
# Visualising evaluation metric score chart - MobileNetV2 baseline
show_performance("mobilenetv2_base", "MobileNetV2 - baseline (frozen backbone)")

#### 2. Cross-Validation & Hyperparameter Tuning

In [ ]:
# ML Model 2 with hyper-parameter optimisation + fine-tuning + cross-validation
step_tune("mobilenetv2")
show_performance("mobilenetv2_tuned", "MobileNetV2 - tuned + fine-tuned")
step_cv("mobilenetv2")

**Which hyperparameter optimization technique have you used and why?** Random search (learning rate, dropout, dense units) for the head, followed by **fine-tuning of the top 30 backbone layers at lr/10** (BatchNorm frozen). Random search is cheap and effective for a few continuous/discrete parameters; the low fine-tuning rate prevents destroying the pretrained features. 3-fold stratified CV verifies stability of the tuned head configuration.

**Have you seen any improvement? Note down the improvement with updated Evaluation metric Score Chart.**

In [ ]:
# Improvement of tuned MobileNetV2 over baseline
step_improvement("mobilenetv2")

#### 3. Explain each evaluation metric's indication towards business and the business impact of the ML model used.
| Metric | What it measures | Business meaning in tumor screening |
|---|---|---|
| **Accuracy** | share of all images classified correctly | Overall reliability; misleading if classes are imbalanced. |
| **Precision (macro)** | of images predicted as a class, how many are right | Low precision = many false alarms → unnecessary follow-up scans, cost and patient anxiety. |
| **Recall (macro)** | of true cases of a class, how many are found | **Most critical:** low recall = missed tumors (delayed treatment, highest clinical and legal risk). |
| **F1-score (macro)** | harmonic mean of precision and recall, averaged over classes | Single balanced score; our model-selection metric because it penalises weak classes. |
| **ROC-AUC (OvR)** | ability to rank the right class above others at any threshold | Shows quality independent of the decision threshold; useful to tune a high-recall operating point. |

*Business impact:* a lightweight, accurate model like MobileNetV2 can pre-sort scans, prioritise suspicious cases for the radiologist and cut first-read time, with low hosting cost; low-confidence predictions are flagged for human review.
### ML Model - 3 : EfficientNetB0 (transfer learning)

In [ ]:
# ML Model 3 - implementation: frozen EfficientNetB0 backbone + new head (baseline)
step_baseline("efficientnetb0")

#### 1. Explain the ML Model used and its performance using Evaluation metric Score Chart.
**EfficientNetB0** is a pretrained CNN designed with *compound scaling* (depth, width and resolution are scaled together) and squeeze-and-excitation blocks. It reaches higher ImageNet accuracy than MobileNetV2 with a moderate size. As in Model 2, the backbone is frozen for the baseline and only the new head is trained.

In [ ]:
# Visualising evaluation metric score chart - EfficientNetB0 baseline
show_performance("efficientnetb0_base", "EfficientNetB0 - baseline (frozen backbone)")

#### 2. Cross-Validation & Hyperparameter Tuning

In [ ]:
# ML Model 3 with hyper-parameter optimisation + fine-tuning + cross-validation
step_tune("efficientnetb0")
show_performance("efficientnetb0_tuned", "EfficientNetB0 - tuned + fine-tuned")
step_cv("efficientnetb0")

**Which hyperparameter optimization technique have you used and why?** The same random search + top-layer fine-tuning + stratified 3-fold CV protocol as for MobileNetV2, so the three models are compared fairly under an identical budget.

**Have you seen any improvement? Note down the improvement with updated Evaluation metric Score Chart.**

In [ ]:
# Improvement of tuned EfficientNetB0 over baseline
step_improvement("efficientnetb0")

### Model comparison and final model selection

In [ ]:
# Compare all 6 models (3 baselines + 3 tuned) + cross-validation summary
rows_cmp = []
for kind in KINDS:
    for stage in ("base", "tuned"):
        r = RESULTS[f"{kind}_{stage}"]
        rows_cmp.append({"model": NAMES[kind], "stage": stage, "val_f1": r["val"]["f1"], **{f"test_{k}": v for k, v in r["test"].items()}})
cmp_df = pd.DataFrame(rows_cmp)
display(cmp_df.round(4))
display(pd.DataFrame(CV_RESULTS).T.round(4).rename_axis("model (3-fold CV, train set)"))

# Efficiency: parameters and inference latency (the brief asks for "most accurate, efficient and reliable")
def latency_ms(model, n=64):
    xs = np.stack([_load(p, 0)[0].numpy() for p in test_df["path"].head(n)])
    model.predict(xs[:8], verbose=0)                       # warm-up
    t0 = time.perf_counter(); model.predict(xs, verbose=0, batch_size=32)
    return (time.perf_counter() - t0) / len(xs) * 1000


eff_df = pd.DataFrame([{"model": NAMES[k], "parameters_M": MODELS[f"{k}_tuned"].count_params() / 1e6,
                        "latency_ms_per_image": latency_ms(MODELS[f"{k}_tuned"])} for k in KINDS])
display(eff_df.round(3))

plot_df = cmp_df.melt(id_vars=["model", "stage"], value_vars=["test_accuracy", "test_precision", "test_recall", "test_f1"])
g = sns.catplot(data=plot_df, x="variable", y="value", hue="model", col="stage", kind="bar", height=4, aspect=1.3)
g.set(ylim=(0, 1.05)); g.fig.suptitle("Test-set metrics of all models", y=1.03); plt.show()

# Model selection on VALIDATION macro-F1 (test set is only used for reporting)
best_key = max([f"{k}_tuned" for k in KINDS], key=lambda n: RESULTS[n]["val"]["f1"])
best_model = MODELS[best_key]
best_kind = best_key.replace("_tuned", "")
print(f"\nSelected final model: {NAMES[best_kind]} (validation macro-F1 = {RESULTS[best_key]['val']['f1']:.4f}, "
      f"test macro-F1 = {RESULTS[best_key]['test']['f1']:.4f})")

#### 1. Which Evaluation metrics did you consider for a positive business impact and why?
**Macro-F1 (primary) with per-class recall and macro-precision as safeguards.** A missed tumor (false negative) is far more harmful than an extra follow-up scan, so recall per class is monitored closely; macro-averaging gives every tumor type equal weight even if one is rare; accuracy alone can hide a weak class; ROC-AUC shows how well the model ranks classes independent of a threshold.

#### 2. Which ML model did you choose from the above created models as your final prediction model and why?
The final model is chosen **programmatically** (cell above): the *tuned* model with the highest **validation macro-F1**. Using the validation set for selection keeps the test set an unbiased estimate of real-world performance. The efficiency table (parameters, latency per image) above supports the choice: a smaller, faster model with similar F1 is preferable for a web app.

#### 3. Explain the model which you have used and the feature importance using any model explainability tool?
**Grad-CAM** (below) computes the gradient of the predicted class score with respect to the last convolutional feature map, weights each channel by its average gradient and overlays the resulting heat map on the MRI. Bright regions are the pixels that most increased the predicted class probability, so we can check that the network looks at the tumor region and not at borders, text marks or background.

In [ ]:
# Explainability: Grad-CAM for the final model
def grad_cam(model, img):
    """Return (heat map in [0,1], predicted class id, probabilities) for one 0-255 image (H, W, 3)."""
    x = tf.convert_to_tensor(img[None, ...], tf.float32)
    with tf.GradientTape() as tape:
        h, conv = x, None
        for layer in model.layers[1:]:
            if isinstance(layer, layers.GlobalAveragePooling2D):   # feature map right before pooling
                conv = h
                tape.watch(conv)
            h = layer(h, training=False)
        cls = int(tf.argmax(h[0]))
        score = h[:, cls]
    grads = tape.gradient(score, conv)
    weights = tf.reduce_mean(grads, axis=(1, 2), keepdims=True)
    cam = tf.nn.relu(tf.reduce_sum(weights * conv, axis=-1))[0].numpy()
    cam = cam / (cam.max() + 1e-8)
    cam = tf.image.resize(cam[..., None], (IMG_SIZE, IMG_SIZE)).numpy()[..., 0]
    return cam, cls, h[0].numpy()


fig, axes = plt.subplots(2, N_CLASSES, figsize=(3.4 * N_CLASSES, 6.4))
axes = np.array(axes).reshape(2, N_CLASSES)
for j, cls_name in enumerate(CLASS_NAMES):
    row = test_df[test_df["label"] == cls_name].iloc[0]
    img = _load(row["path"], 0)[0].numpy()
    cam, pred_id, probs = grad_cam(best_model, img)
    axes[0, j].imshow(img.astype("uint8")); axes[0, j].set_title(f"True: {cls_name}"); axes[0, j].axis("off")
    axes[1, j].imshow(img.astype("uint8")); axes[1, j].imshow(cam, cmap="jet", alpha=0.45)
    axes[1, j].set_title(f"Pred: {CLASS_NAMES[pred_id]} ({probs[pred_id]:.0%})"); axes[1, j].axis("off")
plt.suptitle(f"Grad-CAM - {NAMES[best_kind]}", y=1.0); plt.tight_layout(); plt.show()

## 8. Future Work (Optional)
### 1. Save the best performing ML model in a pickle / joblib / Keras file format for deployment

In [ ]:
# Save the best model (Keras format) + class names (JSON) + metadata (joblib)
os.makedirs("models", exist_ok=True)
best_model.save("models/best_model.keras")
best_model.save("models/best_model.h5")                  # .h5 deliverable required by the brief
for k in KINDS:                                           # one .h5 per model (custom CNN + pretrained models)
    MODELS[f"{k}_tuned"].save(f"models/{k}.h5")
with open("models/class_names.json", "w") as f:
    json.dump(CLASS_NAMES, f)
joblib.dump({"class_names": CLASS_NAMES, "img_size": IMG_SIZE, "best_model": NAMES[best_kind],
             "best_params": BEST_PARAMS[best_kind], "test_metrics": RESULTS[best_key]["test"]}, "models/model_metadata.joblib")
print("Saved:", sorted(os.listdir("models")))

### 2. Load the saved model file again and try to predict unseen data for a sanity check

In [ ]:
# Load the saved model and predict images it has never been trained on (test set)
loaded = keras.models.load_model("models/best_model.keras")
classes_loaded = json.load(open("models/class_names.json"))
check = test_df.sample(min(12, len(test_df)), random_state=SEED).reset_index(drop=True)
imgs = np.stack([_load(p, 0)[0].numpy() for p in check["path"]])
probs = loaded.predict(imgs, verbose=0)
check["predicted"] = [classes_loaded[i] for i in probs.argmax(1)]
check["confidence"] = probs.max(1).round(3)
check["correct"] = check["label"] == check["predicted"]
display(check[["label", "predicted", "confidence", "correct"]])
same = np.allclose(best_model.predict(imgs, verbose=0), probs, atol=1e-4)
loaded_h5 = keras.models.load_model("models/best_model.h5", compile=False)
same_h5 = np.allclose(best_model.predict(imgs, verbose=0), loaded_h5.predict(imgs, verbose=0), atol=1e-4)
print(f"Sanity check accuracy: {check['correct'].mean():.2%} | .keras reload reproduces predictions: {same} | .h5 reload reproduces predictions: {same_h5}")

> **Congrats! Your model is successfully created and ready for deployment on a live server for real user interaction!**
>
> The saved `models/*.h5`, `models/best_model.keras` and `models/class_names.json` are exactly what the Streamlit app (`app.py`) loads: `streamlit run app.py`.
## Conclusion

In [ ]:
# Auto-generated conclusion with the real numbers of this run
t = RESULTS[best_key]["test"]
display(Markdown(f"""
**Summary of this run**

* The cleaned dataset contains **{len(df):,} unique MRI images** in **{N_CLASSES} classes** ({', '.join(CLASS_NAMES)}), split 70/15/15 with stratification.
* Three models were built and tuned with random search and 3-fold cross-validation: a **custom CNN**, **MobileNetV2** and **EfficientNetB0** (transfer learning with fine-tuning).
* The selected final model is **{NAMES[best_kind]}**, with test **accuracy {t['accuracy']:.2%}**, **macro-precision {t['precision']:.2%}**, **macro-recall {t['recall']:.2%}**, **macro-F1 {t['f1']:.2%}** (ROC-AUC {t['roc_auc']:.3f}).
* Grad-CAM shows which regions drive predictions and should be reviewed to confirm the model looks at the lesion.
* All three models are saved as `.h5` (plus `best_model.keras`) in `models/` and are served by the Streamlit app for real-time predictions.

**Business conclusion:** the model can act as a fast first-pass triage aid that prioritises scans and reduces reading time. It is a decision-support prototype, **not a medical device**: low-confidence cases must go to a radiologist, and the model must be validated on external hospital data (and monitored for drift) before any clinical use.

**Limitations & future work:** single-slice 2-D images, one public dataset, no calibration of probabilities, no external validation; next steps are 3-D/multi-slice models, probability calibration, higher input resolution, additional backbones (ResNet50, InceptionV3), ensembling and deployment with monitoring.
"""))

### ***Hurrah! You have successfully completed your Machine Learning Capstone Project!!!***